In [1]:
%pip install -q fastapi uvicorn pyngrok nest-asyncio python-multipart
%pip install -q pandas numpy openpyxl plotly scipy statsmodels requests
%pip install -q "langchain-classic>=1,<2" "langchain-core>=1,<2"
%pip install -q "langchain-community>=0.4,<1" "langchain-huggingface>=1,<2"
%pip install -q "langchain-text-splitters>=1,<2" sentence-transformers faiss-cpu
%pip install -q "transformers>=4.45,<5" accelerate bitsandbytes

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 11.8 MB/s eta 0:00:0000:010:01
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 88.1 MB/s eta 0:00:00:00:010:01
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 84.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.7 MB/s e

## 1. Imports and configuration

In [2]:
import ast
import datetime
import io
import json
import math
import os
import re
import threading
import time
import traceback
import zipfile
from decimal import Decimal
from pathlib import Path
from typing import Optional
from uuid import uuid4

import nest_asyncio
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import requests
import torch
import uvicorn
from fastapi import FastAPI, File, HTTPException, UploadFile
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.exceptions import OutputParserException
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter
from openpyxl import load_workbook
from openpyxl.formula import Tokenizer
from openpyxl.formula.tokenizer import TokenizerError
from openpyxl.utils import column_index_from_string, get_column_letter
from pydantic import BaseModel
from pyngrok import ngrok
from scipy import stats
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

/tmp/ipykernel_49/2098458515.py:28: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
GENERAL_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
CODER_MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
MAX_TOKENS = {"general": 800, "coder": 1500}   # generated-token budget per model
MAX_UPLOAD_BYTES = 15 * 1024 * 1024
MAX_HISTORY_TURNS = 4
MAX_RESULT_ROWS = 200
PREVIEW_ROWS = 5
RETRIEVER_K = 3
MAX_CODE_RETRIES = 1
MAX_FORMULA_RETRIES = 1

app = FastAPI(
    title="Talk to Your Spreadsheet",
    version="9.1"
)
sessions = {}
conversation_memory = {}
model_lock = threading.Lock()

## 2. Load the two quantized models

In [4]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU found. In Kaggle open Session options > Accelerator and choose "
        "GPU T4 x2, then restart the session. 4-bit loading needs CUDA."
    )

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

def load_quantized_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    )
    model.eval()
    return model, tokenizer

general_model, general_tokenizer = load_quantized_model(GENERAL_MODEL_NAME)
coder_model, coder_tokenizer = load_quantized_model(CODER_MODEL_NAME)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [5]:
def generate_text(prompt, max_length=None, model_kind="general", system_message=None):
    if model_kind == "general":
        model, tokenizer = general_model, general_tokenizer
    elif model_kind == "coder":
        model, tokenizer = coder_model, coder_tokenizer
    else:
        raise ValueError(f"Unknown model kind: {model_kind}")

    if max_length is None:
        max_length = MAX_TOKENS[model_kind]
    if system_message is None:
        system_message = (
            "You are a helpful spreadsheet assistant. "
            "Follow the requested instructions and output format exactly."
        )

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generation_kwargs = {
        "max_new_tokens": max_length,
        "num_return_sequences": 1,
        "do_sample": False,
        "use_cache": True,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id
    }
    with model_lock, torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            **generation_kwargs
        )

    input_length = model_inputs.input_ids.shape[1]
    generated_ids = [output_ids[input_length:] for output_ids in generated_ids]

    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

def get_model_response(prompt, model_kind="general", max_length=None):
    responses = generate_text(
        prompt,
        max_length=max_length,
        model_kind=model_kind
    )
    return responses[0].strip()

def extract_json_object(text):
    """Fallback for answers like 'Sure! {...}' that the strict parser rejects."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end <= start:
        raise ValueError("No JSON object found in the model response.")
    return json.loads(text[start:end + 1], strict=False)

def parse_model_response(prompt, output_parser, task_name, model_kind="general"):
    expected_keys = {schema.name for schema in output_parser.response_schemas}
    current_prompt = prompt

    for attempt in range(2):
        raw_response = get_model_response(
            current_prompt,
            model_kind=model_kind
        )
        try:
            try:
                result = output_parser.parse(raw_response)
            except (OutputParserException, ValueError):
                result = extract_json_object(raw_response)
                missing = expected_keys - set(result)
                if missing:
                    raise ValueError(f"Missing keys: {sorted(missing)}")

            if not isinstance(result, dict):
                raise ValueError("Expected a JSON object.")
            return result
        except (OutputParserException, ValueError, KeyError, TypeError) as error:
            if attempt == 1:
                raise ValueError(
                    f"{task_name} returned invalid structured output: {error}"
                ) from error

            current_prompt = (
                prompt
                + "\n\nPrevious response:\n" + raw_response[:1800]
                + "\nError:\n" + str(error)
                + "\nReturn a corrected response using the required JSON schema."
            )

## 3. Structured-output schemas and prompts

In [6]:
intent_schema = ResponseSchema(
    name="intent",
    description=(
        "The user's intent: 'data' for spreadsheet analysis or visualization; "
        "'formula' for generating Excel formulas; "
        "'chat' for greetings, casual conversation or requests unrelated to spreadsheet processing."
    )
)

standalone_question_schema = ResponseSchema(
    name="standalone_question",
    description=(
        "Rewrite a spreadsheet follow-up as a standalone request using conversation history when needed. "
        "For chat messages, return the latest message unchanged."
    )
)

use_previous_result_schema = ResponseSchema(
    name="use_previous_result",
    description=(
        "Whether the request should operate directly on the previous analysis result. "
        "Always false for formula and chat requests."
    ),
    type="boolean"
)

router_parser = StructuredOutputParser.from_response_schemas([
    intent_schema, standalone_question_schema, use_previous_result_schema
])
router_format_instructions = router_parser.get_format_instructions()

objective_schema = ResponseSchema(
    name="objective",
    description="A precise description of the calculation or visualization."
)

plan_columns_schema = ResponseSchema(
    name="columns",
    description="Exact existing DataFrame columns required for the task.",
    type="List[string]"
)

steps_schema = ResponseSchema(
    name="steps",
    description="Ordered, focused steps for the Pandas or Plotly program.",
    type="List[string]"
)

output_schema = ResponseSchema(
    name="output",
    description="Exactly one of: result, fig, both."
)

plan_parser = StructuredOutputParser.from_response_schemas([
    objective_schema, plan_columns_schema, steps_schema, output_schema
])
plan_format_instructions = plan_parser.get_format_instructions()

formula_schema = ResponseSchema(
    name="formula",
    description="One Excel formula beginning with =, or an empty string if impossible."
)

explanation_schema = ResponseSchema(
    name="explanation",
    description="Explanation and correct placement of the Excel formula."
)

columns_used_schema = ResponseSchema(
    name="columns_used",
    description="Exact spreadsheet column names referenced by the formula.",
    type="List[string]"
)

formula_parser = StructuredOutputParser.from_response_schemas([
    formula_schema, explanation_schema, columns_used_schema
])
formula_format_instructions = formula_parser.get_format_instructions()

In [7]:
router_template = """
You are the context-aware router for a spreadsheet assistant.

Classify the LATEST user message into exactly one intent.

Available intents:
- data: The user clearly wants to analyze, calculate, filter, transform, inspect, summarize or visualize spreadsheet data.
- formula: The user clearly wants an Excel formula or function to enter into a worksheet.
- chat: Greetings, thanks, casual conversation, questions about the assistant, or anything that does not clearly require spreadsheet processing.

Requested mode: {requested_mode}
Available worksheets: {sheet_names}

Recent conversation:
{history}

Previous analysis result exists: {has_last_result}
Previous result columns: {previous_result_columns}

Latest user message:
{original_question}

Rules:
1. Classify primarily from the latest message.
2. Do not assume a casual message continues the previous spreadsheet task.
3. Use history only when the latest message clearly depends on earlier context.
4. Greetings, thanks, casual conversation and questions like "what can you do?" are chat.
5. If the message does not clearly request spreadsheet analysis or an Excel formula, choose chat.
6. When uncertain between chat and another intent, choose chat.
7. Choose formula only when the user wants an Excel formula/function.
8. Choose data only when the user wants spreadsheet information, analysis or visualization.
9. For chat, return the latest message unchanged and use_previous_result=false.
10. For formula, use_previous_result=false.
11. use_previous_result=true only when a data request clearly refers to an existing previous result.
12. A new analysis of spreadsheet columns must use_previous_result=false.
13. Do not answer, calculate, generate code or generate formulas.
14. Return only the required structured output.

Examples:
- "Hi" -> chat, false.
- "Thanks!" -> chat, false.
- "How are you?" -> chat, false.
- "What can you do?" -> chat, false.
- "Calculate average Revenue by Region" -> data, false.
- "Which Order_ID values are duplicated?" -> data, false.
- "Now visualize those averages" -> data, true.
- "Now calculate the median instead" -> data, false.
- "Give me an Excel formula for Revenue minus Cost" -> formula, false.

{format_instructions}
"""

router_prompt = PromptTemplate(
    template=router_template,
    input_variables=[
        "requested_mode", "sheet_names", "history",
        "has_last_result", "previous_result_columns", "original_question"
    ],
    partial_variables={"format_instructions": router_format_instructions}
)

chat_template = """
You are the conversational assistant inside Talk to Your Spreadsheet.

Recent conversation:
{history}

User message:
{question}

Respond naturally and briefly.

Rules:
1. Answer the latest message directly.
2. Do not perform spreadsheet analysis or generate Excel formulas in this chat pipeline.
3. Do not claim that you inspected or calculated spreadsheet data.
4. If asked what you can do, briefly explain that you can analyze spreadsheet data, create visualizations and generate Excel formulas.
5. Do not mention internal models, routing or implementation details unless asked.
"""

chat_prompt = PromptTemplate(
    template=chat_template,
    input_variables=["history", "question"]
)

In [8]:
planner_template = """
You are a data-analysis and visualization planner.

User request:
{question}

Available DataFrame columns and types:
{schema}

Sample records:
{sample}

Create a short, precise plan for a Python code-generation model.

Rules:
1. Use only the available columns.
2. If _sheet exists, its exact valid worksheet names are listed in the schema. Use only those exact names.
3. Never invent, rename, singularize or paraphrase worksheet names.
4. Identify the required calculations, filtering, transformations, grouping, aggregation and sorting.
5. If the user requests analysis only, set output to "result".
6. If the user requests visualization only, set output to "fig".
7. If the user requests both, set output to "both".
8. Respect an explicitly requested chart type when appropriate.
9. If no chart type is specified, select the visualization that best communicates the information.
10. Do not generate Python code or calculate the final answer.
11. Return only the required structured output.

{format_instructions}
"""

planner_prompt = PromptTemplate(
    template=planner_template,
    input_variables=["question", "schema", "sample"],
    partial_variables={"format_instructions": plan_format_instructions}
)

coder_template = """
You are an expert Python, Pandas, NumPy and Plotly programmer.

User request:
{question}

Task plan:
{plan}

DataFrame schema:
{schema}

A complete DataFrame named df is already available.

Available objects:
df, pd, np, px, go, stats, math, result, fig.

Rules:
1. Always use the existing df.
2. Never create, recreate, replace or overwrite df.
3. Never create a DataFrame from manually written data.
4. Do not import modules, load files or access external services.
5. Use the exact DataFrame column names.
6. If _sheet exists, its exact valid worksheet names are listed in the schema. Use only those exact values when filtering.
7. Never invent, rename, singularize or paraphrase worksheet names.
8. Use Pandas and NumPy for analysis and Plotly for visualization.
9. If output is "result", assign result only.
10. If output is "fig", assign fig only.
11. If output is "both", assign both result and fig.
12. Never call print(), display() or fig.show().
13. Do not write files.
14. Return ONLY executable Python code.
15. Do not return JSON, explanations or Markdown code fences.
"""

coder_prompt = PromptTemplate(
    template=coder_template,
    input_variables=["question", "plan", "schema"]
)

repair_template = """
You are repairing a Python program for spreadsheet analysis.

Original request:
{question}

Task plan:
{plan}

DataFrame schema:
{schema}

Previous code:
{previous_code}

Execution error:
{error}

Available objects:
df, pd, np, px, go, stats, math, result, fig.

Rules:
1. Always use the existing complete df.
2. Never create, recreate, replace or overwrite df.
3. Never create a DataFrame from manually written data.
4. Do not import modules, access files, access the network or run shell commands.
5. Preserve the original task and task plan.
6. If output is "result", assign result only.
7. If output is "fig", assign fig only.
8. If output is "both", assign both result and fig.
9. Never call print(), display() or fig.show().
10. Return ONLY the complete corrected executable Python program.
11. Do not return JSON, explanations or Markdown code fences.
"""

repair_prompt = PromptTemplate(
    template=repair_template,
    input_variables=["question", "plan", "schema", "previous_code", "error"]
)

In [9]:
formula_template = """
You are an expert Excel formula assistant.

Question:
{question}

Sheet: {sheet_name}
Number of data rows: {n_rows}
Last data row: {last_row}

Actual column mapping:
{column_mapping}

Retrieved Excel documentation:
{context}

Validator feedback on your previous attempt (None on the first attempt):
{feedback}

Generate exactly one Excel formula using the actual column letters.

Rules:
1. Row 1 contains headers. The first data row is row 2.
2. Use row 2 for ordinary per-row calculations.
3. Use another starting row if required, such as row 3 for period-over-period change.
4. Use relative references for formulas intended to be filled down.
5. Use absolute references for fixed ranges.
6. Use actual last-row numbers for bounded ranges.
7. Summary formulas are entered once outside their input range.
8. FILTER, UNIQUE and SORT are dynamic-array formulas entered once.
9. Follow the retrieved syntax, argument order, examples and compatibility notes.
10. Column letters in the documentation are illustrative; replace them using the mapping.
11. Use English Excel function names, comma separators and straight double quotes.
12. Do not reference other worksheets, external files or invented columns.
13. columns_used must contain every column referenced, including columns inside multi-column ranges.
14. Explain where to enter the formula and whether it should be filled down.
15. If the request is impossible, return an empty formula, an explanation and an empty columns_used list.
16. Never write header names inside a formula; use column letters.
17. If validator feedback is given, fix exactly the problems it lists.
18. Return only the requested structured output.

{format_instructions}
"""
formula_prompt = PromptTemplate(
    template=formula_template,
    input_variables=["question", "sheet_name", "n_rows", "last_row", "column_mapping", "context", "feedback"],
    partial_variables={"format_instructions": formula_format_instructions}
)

## 4. RAG: Excel documentation, embeddings and retriever

In [10]:
def find_excel_doc():
    candidates = []
    if os.environ.get("EXCEL_DOC_PATH"):
        candidates.append(Path(os.environ["EXCEL_DOC_PATH"]))
    candidates += [
        Path("/kaggle/input/datasets/ibrahimahmed196/excel-functions/excel_functions.md"),
        Path("/kaggle/working/excel_functions.md"),
        Path("excel_functions.md"),
    ]
    for path in candidates:
        if path.is_file():
            return path

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        found = sorted(kaggle_input.rglob("excel_functions.md"))
        if found:
            return found[0]

    raise FileNotFoundError(
        "excel_functions.md was not found. Attach the 'excel-functions' dataset "
        "(Add Input) or upload the file next to this notebook."
    )

EXCEL_DOC_PATH = find_excel_doc()
excel_documentation = EXCEL_DOC_PATH.read_text(encoding="utf-8")
print("Using documentation file:", EXCEL_DOC_PATH)

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("##", "title")],
    strip_headers=False
)
section_documents = markdown_splitter.split_text(excel_documentation)
excel_documents = []

for document in section_documents:
    title = document.metadata.get("title")
    if not title:
        continue

    source_match = re.search(r"(?m)^Source:\s*(.+)$", document.page_content)
    document.metadata["source"] = (source_match.group(1).strip() if source_match else str(EXCEL_DOC_PATH))
    document.metadata["document_type"] = ("recipe" if title.lower().startswith("recipe:") else "function")
    document.page_content = re.sub(r"(?m)^Source:.*(?:\n|$)", "", document.page_content).strip()

    if document.page_content:
        excel_documents.append(document)

print(f"Loaded {len(excel_documents)} documentation sections.")

Using documentation file: /kaggle/input/datasets/ibrahimahmed196/excel-functions/excel_functions.md
Loaded 52 documentation sections.


In [11]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = FAISS.from_documents(
    documents=excel_documents,
    embedding=embedding_model
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": RETRIEVER_K}
)

def retrieve_excel_docs(question):
    return retriever.invoke(question)

def strip_heading(text):
    """The first line of each section is its '## Title' heading; the title is shown separately."""
    return re.sub(r"\A##[^\n]*\n", "", text).strip()

def format_retrieved_docs(documents):
    return "\n\n".join(
        f"{doc.metadata.get('title', 'Excel documentation')}\n{strip_heading(doc.page_content)}"
        for doc in documents
    ) or "No documentation was retrieved."

def retrieved_docs_to_json(documents):
    return [
        {
            "title": doc.metadata.get("title", ""),
            "source": doc.metadata.get("source", ""),
            "text": strip_heading(doc.page_content)
        }
        for doc in documents
    ]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 5. Chains, conversation memory and routing

In [12]:
def make_structured_chain(prompt, parser, task_name, model_kind="general"):
    return prompt | RunnableLambda(
        lambda value: parse_model_response(
            value.to_string(), parser, task_name, model_kind=model_kind
        )
    )

router_chain = make_structured_chain(router_prompt, router_parser, "Intent routing")
planner_chain = make_structured_chain(planner_prompt, plan_parser, "Task planning")
formula_chain = make_structured_chain(formula_prompt, formula_parser, "Excel formula generation")

coder_llm = RunnableLambda(
    lambda value: get_model_response(value.to_string(), model_kind="coder")
)
coder_chain = coder_prompt | coder_llm
repair_chain = repair_prompt | coder_llm

general_llm = RunnableLambda(
    lambda value: get_model_response(value.to_string(), model_kind="general")
)
chat_chain = chat_prompt | general_llm

def create_conversation(session_id):
    conversation_memory[session_id] = {"history": [], "last_result": None}

def get_conversation(session_id):
    if session_id not in conversation_memory:
        create_conversation(session_id)
    return conversation_memory[session_id]

def remember_turn(session_id, question, resolved_question, mode, summary):
    memory = get_conversation(session_id)
    memory["history"].append({
        "question": question,
        "resolved_question": resolved_question,
        "mode": mode,
        "summary": summary[:500]
    })
    memory["history"] = memory["history"][-MAX_HISTORY_TURNS:]

def format_history(session_id):
    history = get_conversation(session_id)["history"]
    if not history:
        return "No previous conversation."

    return "\n\n".join(
        f"User: {turn['question']}\n"
        f"Resolved: {turn['resolved_question']}\n"
        f"Intent: {turn['mode']}\n"
        f"Assistant: {turn['summary']}"
        for turn in history
    )

def unique_names(names):
    seen, result = {}, []
    for name in names:
        name = str(name)
        count = seen.get(name, 0)
        seen[name] = count + 1
        result.append(name if count == 0 else f"{name}_{count + 1}")
    return result

def to_display_frame(result):
    if isinstance(result, pd.Series):
        result = result.to_frame(name=result.name if result.name is not None else "Value")

    frame = result.copy()

    if isinstance(frame.columns, pd.MultiIndex):
        frame.columns = [
            " ".join(str(part) for part in column if str(part) != "").strip()
            for column in frame.columns
        ]
    else:
        frame.columns = [str(column) for column in frame.columns]

    index = frame.index
    meaningless = (
        isinstance(index, pd.RangeIndex)
        or (index.nlevels == 1 and index.name is None and pd.api.types.is_integer_dtype(index))
    )

    if meaningless:
        frame = frame.reset_index(drop=True)
    else:
        existing = set(frame.columns)
        frame.index.names = [
            str(name) if name is not None else ("Item" if index.nlevels == 1 else f"Level {i + 1}")
            for i, name in enumerate(index.names)
        ]
        frame.index.names = [
            f"{name} (index)" if name in existing else name
            for name in frame.index.names
        ]
        frame = frame.reset_index()

    frame.columns = unique_names(frame.columns)
    return frame

def result_to_dataframe(result):
    try:
        if isinstance(result, (pd.DataFrame, pd.Series)):
            return to_display_frame(result)

        if isinstance(result, dict) and result:
            if all(np.isscalar(v) for v in result.values()):
                return pd.DataFrame({"Group": list(result.keys()), "Value": list(result.values())})
            return to_display_frame(pd.DataFrame(result))

        if isinstance(result, list) and result and isinstance(result[0], dict):
            return pd.DataFrame(result)
    except (ValueError, TypeError):
        return None

    return None

def save_analysis_result(session_id, result):
    result_df = result_to_dataframe(result)
    get_conversation(session_id)["last_result"] = (
        result_df if result_df is not None and not result_df.empty else None
    )

def as_bool(value):
    if isinstance(value, bool):
        return value
    if value is None:
        return False
    if isinstance(value, (int, float)) and value in (0, 1):
        return bool(value)
    if isinstance(value, str):
        text = value.strip().lower()
        if text in {"true", "yes", "1"}:
            return True
        if text in {"false", "no", "0", "", "none", "null"}:
            return False
    raise ValueError(f"use_previous_result must be a boolean, got {value!r}.")

def select_intent(session_id, original_question, requested_mode, workbook):
    requested_mode = requested_mode.strip().lower()
    if requested_mode not in {"auto", "data", "formula", "chat"}:
        raise ValueError(f"Invalid mode: {requested_mode}")

    memory = get_conversation(session_id)
    last_result = memory["last_result"]

    if requested_mode != "auto" and not memory["history"]:
        return {
            "intent": requested_mode,
            "standalone_question": original_question,
            "use_previous_result": False,
            "routed": False
        }

    decision = router_chain.invoke({
        "requested_mode": requested_mode,
        "sheet_names": ", ".join(workbook.keys()),
        "history": format_history(session_id),
        "has_last_result": last_result is not None,
        "previous_result_columns": list(last_result.columns) if last_result is not None else [],
        "original_question": original_question
    })

    intent_text = str(decision.get("intent", "")).strip().lower()
    standalone_question = decision.get("standalone_question")

    if "formula" in intent_text:
        intent = "formula"
    elif intent_text in {"data", "analysis", "visualization", "visualisation", "chart"}:
        intent = "data"
    elif intent_text in {"chat", "general", "conversation", "casual"}:
        intent = "chat"
    else:
        intent = "chat"

    if requested_mode != "auto":
        intent = requested_mode

    if intent == "chat":
        standalone_question = original_question.strip()
        use_previous_result = False
    else:
        if not isinstance(standalone_question, str) or not standalone_question.strip():
            raise ValueError("The router returned an invalid standalone question.")

        use_previous_result = as_bool(decision.get("use_previous_result"))
        if intent == "formula" or last_result is None:
            use_previous_result = False

    return {
        "intent": intent,
        "standalone_question": standalone_question.strip(),
        "use_previous_result": use_previous_result,
        "routed": requested_mode == "auto"
    }

## 6. Excel formula validation

In [13]:
ALLOWED_FUNCTIONS = {
    # aggregation
    "SUM", "SUMIF", "SUMIFS", "SUMPRODUCT",
    "AVERAGE", "AVERAGEIF", "AVERAGEIFS", "MEDIAN", "STDEV.S", "STDEV.P",
    "COUNT", "COUNTA", "COUNTBLANK", "COUNTIF", "COUNTIFS",
    "MAX", "MIN", "MAXIFS", "MINIFS", "LARGE", "SMALL",
    # maths and rounding
    "ROUND", "ROUNDUP", "ROUNDDOWN", "ABS", "MOD", "INT", "SQRT", "POWER",
    # logic
    "IF", "IFS", "IFERROR", "IFNA", "AND", "OR", "NOT",
    "ISBLANK", "ISNUMBER", "ISTEXT", "ISERROR",
    # lookup and ranking
    "RANK", "RANK.EQ", "VLOOKUP", "XLOOKUP", "INDEX", "MATCH",
    # text
    "LEFT", "RIGHT", "MID", "LEN", "TRIM", "UPPER", "LOWER", "PROPER",
    "CONCAT", "TEXTJOIN", "TEXT", "SUBSTITUTE", "FIND", "SEARCH", "VALUE",
    # dates
    "YEAR", "MONTH", "DAY", "DATE", "TODAY", "DATEDIF", "EOMONTH", "WEEKDAY",
    # dynamic arrays
    "FILTER", "UNIQUE", "SORT"
}

class FormulaUnavailable(ValueError):
    """The model reports that the request cannot be written as one formula."""

CELL_REFERENCE = re.compile(r"^(\$?)([A-Za-z]{1,3})(\$?)([1-9][0-9]*)$")
COLUMN_REFERENCE = re.compile(r"^\$?([A-Za-z]{1,3})$")

def build_column_mapping(df):
    return "\n".join(
        f"{get_column_letter(i + 1)} | {column}"
        for i, column in enumerate(df.columns)
    )

def parse_reference(reference, n_columns):
    """Return (column letters used, row info for a multi-row range or None)."""
    if any(c in reference for c in "![]"):
        raise ValueError("External or cross-sheet references are unsupported.")

    parts = reference.split(":")
    if len(parts) > 2:
        raise ValueError(f"Invalid reference: {reference}")

    endpoints = []
    for part in parts:
        cell = CELL_REFERENCE.fullmatch(part)
        column = COLUMN_REFERENCE.fullmatch(part)
        if cell is None and column is None:
            raise ValueError(
                f"Invalid reference '{reference}'. Use column letters such as "
                f"B2:B100, never header names."
            )

        letters = (cell.group(2) if cell else column.group(1)).upper()
        index = column_index_from_string(letters)
        if index > n_columns:
            raise ValueError(
                f"Column {letters} does not exist; the worksheet has {n_columns} columns."
            )

        row = int(cell.group(4)) if cell else None
        if row is not None and row > 1048576:
            raise ValueError(f"Invalid row: {row}")
        endpoints.append((index, row, bool(cell and cell.group(3))))

    if len(endpoints) == 1:
        return [get_column_letter(endpoints[0][0])], None

    (start_col, start_row, start_abs), (end_col, end_row, end_abs) = endpoints
    if start_col > end_col or (start_row is None) != (end_row is None):
        raise ValueError(f"Invalid range: {reference}")
    if start_row is not None and start_row > end_row:
        raise ValueError(f"Reversed row range: {reference}")

    letters = [get_column_letter(i) for i in range(start_col, end_col + 1)]
    row_info = (start_row, end_row, start_abs, end_abs) if start_row is not None else None
    return letters, row_info

def validate_excel_formula(answer, df):
    formula = answer.get("formula")
    explanation = answer.get("explanation")

    if not isinstance(formula, str) or not isinstance(explanation, str):
        raise ValueError("Formula and explanation must be strings.")

    formula = formula.strip().strip("`").strip()
    if not formula:
        raise FormulaUnavailable(explanation or "No formula could be generated.")
    if not formula.startswith("="):
        formula = "=" + formula          # the model often drops the leading =
    if len(formula) > 1000 or "\n" in formula:
        raise ValueError("Invalid formula format.")

    try:
        tokens = Tokenizer(formula).items
    except (TokenizerError, IndexError, ValueError) as error:
        raise ValueError(f"Formula tokenization failed: {error}") from error

    if not tokens:
        raise ValueError("The formula is empty.")

    n_columns = len(df.columns)
    last_row = len(df) + 1
    stack = []
    letters = set()
    ranges = []

    for token in tokens:
        if token.type in {"FUNC", "PAREN"} and token.subtype == "OPEN":
            stack.append(token.type)
        elif token.type in {"FUNC", "PAREN"} and token.subtype == "CLOSE":
            if not stack or stack.pop() != token.type:
                raise ValueError("Mismatched parentheses.")

        if token.type == "ARRAY":
            raise ValueError("Array constants are unsupported.")

        if token.type == "FUNC" and token.subtype == "OPEN":
            name = token.value[:-1].upper()
            if name not in ALLOWED_FUNCTIONS:
                raise ValueError(f"Unsupported Excel function: {name}")

        if token.type == "OPERAND" and token.subtype == "RANGE":
            used, row_info = parse_reference(token.value, n_columns)
            letters.update(used)
            if row_info:
                ranges.append((token.value, row_info))

        if token.type == "OPERAND" and token.subtype == "ERROR":
            raise ValueError(f"Excel error literal: {token.value}")

        if token.type == "SEP" and token.subtype == "ROW":
            raise ValueError("Array row separators are unsupported.")

    if stack or tokens[-1].type in {
        "OPERATOR-INFIX", "OPERATOR-PREFIX", "SEP"
    } or tokens[0].type == "OPERATOR-INFIX":
        raise ValueError("Incomplete or unbalanced formula.")

    # A fixed range that stops early silently drops data (typically because the
    # documentation's illustrative row 100 was copied). Expanding ranges such as
    # $H$2:H2 are fine and are skipped.
    for reference, (start_row, end_row, start_abs, end_abs) in ranges:
        expanding = start_abs and not end_abs
        if (start_row == 2 and end_row - start_row >= 9
                and end_row < last_row and not expanding):
            raise ValueError(
                f"Range {reference} stops at row {end_row} but the data "
                f"continues to row {last_row}. Use row {last_row} as the last row."
            )

    letter_to_name = {
        get_column_letter(i + 1): str(name)
        for i, name in enumerate(df.columns)
    }

    return {
        "formula": formula,
        "explanation": explanation,
        "columns_used": [
            letter_to_name[letter]
            for letter in sorted(letters, key=column_index_from_string)
        ],
        "column_mapping": {
            letter: letter_to_name[letter]
            for letter in sorted(letters, key=column_index_from_string)
        }
    }

def generate_excel_formula(question, df, sheet_name, use_rag=True):
    documents = retrieve_excel_docs(question) if use_rag else []
    feedback = "None"

    for attempt in range(MAX_FORMULA_RETRIES + 1):
        answer = formula_chain.invoke({
            "question": question,
            "sheet_name": sheet_name,
            "n_rows": len(df),
            "last_row": len(df) + 1,
            "column_mapping": build_column_mapping(df),
            "context": format_retrieved_docs(documents),
            "feedback": feedback
        })

        try:
            validated = validate_excel_formula(answer, df)
            break
        except FormulaUnavailable:
            raise
        except ValueError as error:
            if attempt == MAX_FORMULA_RETRIES:
                raise
            feedback = (
                f"The formula {answer.get('formula')!r} was rejected: {error} "
                "Return a corrected formula."
            )

    return {
        "answer": validated,
        "retrieved": retrieved_docs_to_json(documents)
    }

## 7. Planner, code generation and the restricted Python runner

In [14]:
def dataframe_context(df):
    schema = "\n".join(f"{column}: {df[column].dtype}" for column in df.columns)

    if "_sheet" in df.columns:
        sheet_names = [str(name) for name in df["_sheet"].dropna().unique()]
        schema += "\nExact worksheet names in _sheet: " + ", ".join(sheet_names)

    sample = df.head(3).to_json(
        orient="records",
        date_format="iso",
        force_ascii=False,
        default_handler=str
    )

    return schema, sample

def normalize_plan_output(value):
    text = value.strip().lower()
    wants_fig = any(word in text for word in ("fig", "chart", "plot", "visual"))
    wants_result = "result" in text

    if "both" in text or (wants_fig and wants_result):
        return "both"
    if wants_fig:
        return "fig"
    if wants_result:
        return "result"
    raise ValueError("Output must be result, fig or both.")

def create_task_plan(question, df, intent="data"):
    if intent != "data":
        raise ValueError("The planner only handles data requests.")

    schema, sample = dataframe_context(df)
    plan = planner_chain.invoke({"question": question, "schema": schema, "sample": sample})

    if not isinstance(plan.get("objective"), str):
        raise ValueError("The plan requires an objective.")
    if not isinstance(plan.get("columns"), list):
        raise ValueError("The plan requires a column list.")
    if not isinstance(plan.get("steps"), list) or not plan["steps"]:
        raise ValueError("The plan requires execution steps.")

    by_lowercase = {str(c).strip().lower(): c for c in df.columns}
    known, derived = [], []

    for name in plan["columns"]:
        match = name if name in df.columns else by_lowercase.get(name.strip().lower())
        if match is None:
            derived.append(name)
        else:
            known.append(match)

    plan["columns"] = list(dict.fromkeys(known))
    if derived:
        plan["derived_columns"] = derived

    plan["output"] = normalize_plan_output(plan["output"])
    return plan

def clean_python_code(code):
    code = code.strip()
    if code.startswith("```"):
        lines = code.splitlines()[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        code = "\n".join(lines)
    return code.strip()

def generate_python_code(question, df, plan):
    schema, _ = dataframe_context(df)

    code = coder_chain.invoke({
        "question": question,
        "plan": json.dumps(plan, ensure_ascii=False),
        "schema": schema
    })

    code = clean_python_code(code)
    if not code:
        raise ValueError("The code model did not return Python code.")

    return {"code": code, "explanation": plan["objective"]}

def repair_python_code(question, df, plan, previous_code, error):
    schema, _ = dataframe_context(df)

    code = repair_chain.invoke({
        "question": question,
        "plan": json.dumps(plan, ensure_ascii=False),
        "schema": schema,
        "previous_code": previous_code,
        "error": error[:1200]
    })

    code = clean_python_code(code)
    if not code:
        raise ValueError("The repair model did not return Python code.")

    return {"code": code, "explanation": plan["objective"]}

In [15]:
def _ignore(*args, **kwargs):
    return None

SAFE_BUILTINS = {
    "len": len, "range": range, "list": list, "dict": dict, "tuple": tuple,
    "set": set, "str": str, "int": int, "float": float, "bool": bool,
    "abs": abs, "min": min, "max": max, "sum": sum, "round": round,
    "sorted": sorted, "enumerate": enumerate, "zip": zip,
    "any": any, "all": all, "isinstance": isinstance, "print": _ignore
}

def prepare_generated_code(code):
    code = clean_python_code(code)

    cleaned_lines = []
    for line in code.splitlines():
        stripped = line.strip()
        if stripped.startswith("import ") or stripped.startswith("from "):
            continue
        cleaned_lines.append(line)

    code = "\n".join(cleaned_lines).strip()

    if re.search(r"^\s*df\s*=", code, flags=re.MULTILINE):
        raise ValueError("Generated code attempted to overwrite df.")

    return code

def run_generated_code(code, df, expected_output):
    code = prepare_generated_code(code)
    if not code:
        raise ValueError("Generated Python code is empty.")

    scope = {
        "__builtins__": SAFE_BUILTINS,
        "df": df.copy(), "pd": pd, "np": np, "px": px, "go": go,
        "stats": stats, "math": math, "result": None, "fig": None
    }

    exec(code, scope, scope)
    result, fig = scope["result"], scope["fig"]

    if expected_output in {"result", "both"} and result is None:
        raise ValueError("Generated code did not assign result.")
    if expected_output in {"fig", "both"} and fig is None:
        raise ValueError("Generated code did not assign fig.")

    return result, fig, code

def execute_code_pipeline(question, df, intent):
    plan = create_task_plan(question, df, intent)
    generated = generate_python_code(question, df, plan)

    for attempt in range(MAX_CODE_RETRIES + 1):
        try:
            result, fig, code = run_generated_code(generated["code"], df, plan["output"])
            return {
                "plan": plan, "code": code, "explanation": generated["explanation"],
                "result": result, "fig": fig, "attempts": attempt + 1
            }
        except Exception as error:
            if attempt == MAX_CODE_RETRIES:
                raise ValueError(
                    f"Code execution failed after {attempt + 1} attempts: "
                    f"{type(error).__name__}: {error}"
                ) from error

            generated = repair_python_code(
                question=question, df=df, plan=plan,
                previous_code=generated["code"],
                error=f"{type(error).__name__}: {error}"
            )

## 8. Result serialization

In [16]:
def make_json_safe(value):
    if value is None or value is pd.NA or value is pd.NaT:
        return None
    if isinstance(value, np.generic):
        return make_json_safe(value.item())
    if isinstance(value, (datetime.datetime, datetime.date)):   
        return value.isoformat()
    if isinstance(value, (datetime.timedelta, pd.Period, pd.Interval)):
        return str(value)
    if isinstance(value, Decimal):
        return float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, dict):
        return {str(k): make_json_safe(v) for k, v in value.items()}
    if isinstance(value, (pd.Series, pd.Index, pd.Categorical, np.ndarray)):
        return [make_json_safe(v) for v in value.tolist()]
    if isinstance(value, (list, tuple, set, frozenset)):
        return [make_json_safe(v) for v in value]
    return str(value)   

def serialize_result(result):
    if result is None:
        return None

    if isinstance(result, (pd.Series, pd.DataFrame)):
        frame = to_display_frame(result)
        preview = frame.head(MAX_RESULT_ROWS)
        return {
            "kind": "table",
            "columns": [str(c) for c in preview.columns],
            "rows": make_json_safe(preview.to_dict(orient="records")),
            "total_rows": len(frame),
            "truncated": len(frame) > MAX_RESULT_ROWS
        }

    if isinstance(result, dict):
        return {"kind": "object", "value": make_json_safe(result)}

    if isinstance(result, (list, tuple, set, np.ndarray, pd.Index, pd.Categorical)):
        values = make_json_safe(result)
        return {
            "kind": "list",
            "value": values[:MAX_RESULT_ROWS],
            "truncated": len(values) > MAX_RESULT_ROWS
        }

    return {"kind": "scalar", "value": make_json_safe(result)}

## 9. API endpoints

In [17]:
def column_kind(series):
    if pd.api.types.is_bool_dtype(series):
        return "boolean"
    if pd.api.types.is_numeric_dtype(series):
        return "number"
    if pd.api.types.is_datetime64_any_dtype(series):
        return "date"
    return "text"

def read_workbook(contents):
    """Read every worksheet, validate row-1 headers and return ({name: DataFrame}, [skipped])."""
    raw = load_workbook(io.BytesIO(contents), read_only=True, data_only=True)
    try:
        headers_by_sheet = {}
        for sheet in raw.worksheets:
            first_row = next(
                sheet.iter_rows(min_row=1, max_row=1, values_only=True),
                None
            )
            headers_by_sheet[sheet.title] = list(first_row) if first_row else []
    finally:
        raw.close()

    frames = pd.read_excel(io.BytesIO(contents), sheet_name=None, engine="openpyxl")
    sheets, skipped = {}, []

    for name, df in frames.items():
        headers = headers_by_sheet.get(name, [])
        while headers and (headers[-1] is None or str(headers[-1]).strip() == ""):
            headers.pop()

        if not headers:
            skipped.append(name)          
            continue

        blank = [
            get_column_letter(i + 1)
            for i, h in enumerate(headers)
            if h is None or str(h).strip() == ""
        ]
        if blank:
            raise ValueError(
                f"Sheet '{name}': row 1 has no header in column {', '.join(blank)}. "
                "Every column needs a header in row 1."
            )

        names = [str(h).strip() for h in headers]   # numeric headers such as 2024 become text
        duplicates = sorted({n for n in names if names.count(n) > 1})
        if duplicates:
            raise ValueError(f"Sheet '{name}': duplicate headers: {', '.join(duplicates)}.")
        if "_sheet" in names:
            raise ValueError("'_sheet' is a reserved column name.")

        if len(df.columns) < len(names):
            raise ValueError(f"Sheet '{name}' could not be read consistently.")
        df = df.iloc[:, :len(names)].copy()
        df.columns = names
        filled = df.notna().any(axis=1)
        df = df.loc[:filled[filled].index.max()] if filled.any() else df.iloc[0:0]
        sheets[name] = df

    if not sheets:
        raise ValueError("The workbook has no worksheet with a header row in row 1.")
    return sheets, skipped

@app.get("/health")
def health():
    return {
        "status": "running",
        "sessions": len(sessions),
        "general_model": GENERAL_MODEL_NAME,
        "coder_model": CODER_MODEL_NAME,
        "rag_documents": vectorstore.index.ntotal
    }

@app.post("/upload")
async def upload_excel(file: UploadFile = File(...)):
    filename = file.filename or ""
    if not filename.lower().endswith(".xlsx"):
        raise HTTPException(400, "Only .xlsx workbooks are supported.")

    contents = await file.read(MAX_UPLOAD_BYTES + 1)
    if len(contents) > MAX_UPLOAD_BYTES:
        raise HTTPException(413, "File exceeds the 15 MB limit.")

    try:
        sheets, skipped = read_workbook(contents)

        session_id = str(uuid4())
        sessions[session_id] = sheets
        create_conversation(session_id)

        return {
            "status": "success",
            "session_id": session_id,
            "filename": filename,
            "skipped_sheets": skipped,
            "sheets": {
                name: {
                    "rows": len(df),
                    "columns": len(df.columns),
                    "column_names": list(df.columns),
                    "column_kinds": {c: column_kind(df[c]) for c in df.columns},
                    "column_unique": {c: int(df[c].nunique()) for c in df.columns},
                    "preview": make_json_safe(
                        df.head(PREVIEW_ROWS).to_dict(orient="records")
                    )
                }
                for name, df in sheets.items()
            }
        }

    except zipfile.BadZipFile as error:
        raise HTTPException(
            400, "This file is not a valid .xlsx workbook."
        ) from error

    except Exception as error:
        raise HTTPException(400, str(error)) from error

@app.get("/history/{session_id}")
def get_history(session_id: str):
    if session_id not in sessions:
        raise HTTPException(404, "Workbook session not found.")

    memory = get_conversation(session_id)
    return {
        "session_id": session_id,
        "history": memory["history"],
        "has_last_result": memory["last_result"] is not None
    }

@app.post("/reset/{session_id}")
def reset_conversation(session_id: str):
    """Forget the conversation (history and last result) but keep the workbook."""
    if session_id not in sessions:
        raise HTTPException(404, "Workbook session not found.")
    create_conversation(session_id)
    return {"status": "reset"}

In [18]:
class QuestionRequest(BaseModel):
    session_id: str
    question: str
    mode: str = "auto"
    sheet_name: Optional[str] = None
    use_rag: bool = True

def select_dataframe(workbook, sheet_name):
    if sheet_name is not None:
        if sheet_name not in workbook:
            raise ValueError(f"Unknown worksheet: {sheet_name}")
        return workbook[sheet_name], "sheet"

    if len(workbook) == 1:
        return next(iter(workbook.values())), "sheet"

    frames = []
    for name, df in workbook.items():
        temp = df.copy()
        temp["_sheet"] = name
        frames.append(temp)

    return pd.concat(frames, ignore_index=True), "workbook"

@app.post("/ask")
def ask(data: QuestionRequest):
    started = time.perf_counter()
    question = data.question.strip()

    if not question:
        raise HTTPException(400, "Question cannot be empty.")
    if data.session_id not in sessions:
        raise HTTPException(404, "Workbook session not found.")

    workbook = sessions[data.session_id]
    memory = get_conversation(data.session_id)

    try:
        routing = select_intent(
            session_id=data.session_id,
            original_question=question,
            requested_mode=data.mode,
            workbook=workbook
        )

        mode = routing["intent"]
        resolved = routing["standalone_question"]

        if mode == "chat":
            response = chat_chain.invoke({
                "question": resolved,
                "history": format_history(data.session_id)
            }).strip()

            remember_turn(data.session_id, question, resolved, mode, response)

            return {
                "session_id": data.session_id,
                "question": question,
                "resolved_question": resolved,
                "mode": "chat",
                "router": routing,
                "result": response,
                "elapsed": round(time.perf_counter() - started, 1)
            }

        if mode == "formula":
            sheet_name = data.sheet_name or next(iter(workbook))
            if sheet_name not in workbook:
                raise ValueError(f"Unknown worksheet: {sheet_name}")

            generated = generate_excel_formula(
                question=resolved,
                df=workbook[sheet_name],
                sheet_name=sheet_name,
                use_rag=data.use_rag
            )

            answer = generated["answer"]
            remember_turn(
                data.session_id, question, resolved, mode,
                f"Generated formula: {answer['formula']}. {answer['explanation']}"
            )

            return {
                "session_id": data.session_id,
                "question": question,
                "resolved_question": resolved,
                "mode": "formula",
                "router": routing,
                "sheet_name": sheet_name,
                "use_rag": data.use_rag,
                "retrieved": generated["retrieved"],
                "result": answer,
                "elapsed": round(time.perf_counter() - started, 1)
            }

        if routing["use_previous_result"]:
            df = memory["last_result"].copy()
            source = "last_result"
        else:
            df, source = select_dataframe(workbook, data.sheet_name)

        generated = execute_code_pipeline(question=resolved, df=df, intent="data")
        raw_result, figure = generated["result"], generated["fig"]
        serialized_result = serialize_result(raw_result)

        if raw_result is not None:
            save_analysis_result(data.session_id, raw_result)
        elif source != "last_result":
            memory["last_result"] = None

        summary = (
            f"Performed data task: {generated['plan']['objective']}. "
            f"Result: {str(serialized_result)[:250]}"
        )
        remember_turn(data.session_id, question, resolved, mode, summary)

        return {
            "session_id": data.session_id,
            "question": question,
            "resolved_question": resolved,
            "mode": "data",
            "router": routing,
            "sheet_name": data.sheet_name,
            "source": source,
            "plan": generated["plan"],
            "code": generated["code"],
            "explanation": generated["explanation"],
            "attempts": generated["attempts"],
            "result": serialized_result,
            "figure": pio.to_json(figure) if figure is not None else None,
            "elapsed": round(time.perf_counter() - started, 1)
        }

    except ValueError as error:
        raise HTTPException(400, str(error)) from error
    except Exception as error:
        traceback.print_exc()
        raise HTTPException(500, "Request failed. Check the Kaggle backend logs.") from error

## 10. Start the server and open the ngrok tunnel

In [ ]:
nest_asyncio.apply()
LOCAL_URL = "http://127.0.0.1:8000"

def backend_is_up():
    try:
        return requests.get(f"{LOCAL_URL}/health", timeout=2).status_code == 200
    except requests.RequestException:
        return False

def run_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )

if backend_is_up():
    print("A server is already running on port 8000.")
else:
    server_thread = threading.Thread(
        target=run_server,
        daemon=True
    )
    server_thread.start()

    for attempt in range(20):
        if backend_is_up():
            break
        time.sleep(1)
    else:
        raise RuntimeError("FastAPI did not start.")

try:
    from kaggle_secrets import UserSecretsClient
    NGROK_TOKEN = UserSecretsClient().get_secret("NGROK_TOKEN")
except Exception:
    NGROK_TOKEN = os.environ.get("NGROK_TOKEN", "YOUR_NGROK_TOKEN")

if not NGROK_TOKEN or NGROK_TOKEN == "YOUR_NGROK_TOKEN":
    raise ValueError("Add your ngrok authtoken as a Kaggle secret named NGROK_TOKEN.")

ngrok.set_auth_token(NGROK_TOKEN)

for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

public_url = ngrok.connect(8000).public_url

print("Backend URL:", public_url)
print("Paste this URL into the frontend script (Backend > URL).")
print(requests.get(f"{public_url}/health", timeout=30).json())